In [3]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.path.abspath('../src')) # Using absolute path is safer

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
%load_ext autoreload
%autoreload 2

import polars as pl
import numpy as np
from datetime import datetime, timedelta
import os
# setup paths

RAW_FOLDER = "../data/raw/"
MASTER_PATH = "../data/raw/mind1_chaos_master.csv"

# load one of your kaggle files eg (electronics)
file_name = "D:/Mind_1_Project/data/raw/us-shein-electronics-4395.csv"
df = pl.read_csv(os.path.join(RAW_FOLDER, file_name))

# Inject the synthetic "Messy" dates
def generate_messy_date():
    if np.random.rand() < 0.1:
        return None
    dt = datetime(2026, 1, 1) + timedelta(days=np.random.randint(0, 60))
    chance = np.random.rand()
    if chance < 0.4:
        return dt.strftime("%Y-%m-%d")
    elif chance < 0.7:
        return dt.strftime("%d/%m/%Y")
    else:
        return dt.strftime("%b %d, %Y")

messy_dates = [generate_messy_date() for _ in range(df.height)]
df = df.with_columns(pl.Series("date_unfixed", messy_dates))

df.write_csv(MASTER_PATH)

print("Successfully created chaos master with {df.height} rows.")
print("Columns availble for Mind_1 to clean: ")
print(df.columns)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Successfully created chaos master with {df.height} rows.
Columns availble for Mind_1 to clean: 
['goods-title-link--jump', 'goods-title-link--jump href', 'rank-title', 'rank-sub', 'price', 'discount', 'selling_proposition', 'color-count', 'goods-title-link', 'date_unfixed']


In [5]:
import sys
sys.path.append('../src') # allows notebook to see your folders
from actions import CleaningActions
import polars as pl

#load the chaos_master
df = pl.read_csv("../data/raw/mind1_chaos_master.csv")

# test the currency stripper on the price column 
df_test = CleaningActions.strip_currency(df, "price")

# Test the data Unifier on our date_unfixed column
df_test = CleaningActions.unify_date(df_test, "date_unfixed")

# show the result 
print("-----Before---")
print(df.select(["price", "date_unfixed"]).head(5))

print("\n---After---")
print(df_test.select(["price", "date_unfixed"]).head(5))

# Check the price if it is anumber now or not 
print(f"\nPrice Column Type : {df_test['price'].dtype}")


-----Before---
shape: (5, 2)
┌───────┬──────────────┐
│ price ┆ date_unfixed │
│ ---   ┆ ---          │
│ str   ┆ str          │
╞═══════╪══════════════╡
│ $1.36 ┆ 2026-01-11   │
│ $2.32 ┆ 2026-02-26   │
│ $1.70 ┆ Jan 30, 2026 │
│ $0.38 ┆ 18/02/2026   │
│ $3.10 ┆ 2026-01-26   │
└───────┴──────────────┘

---After---
shape: (5, 2)
┌───────┬──────────────┐
│ price ┆ date_unfixed │
│ ---   ┆ ---          │
│ f64   ┆ date         │
╞═══════╪══════════════╡
│ 1.36  ┆ 2026-01-11   │
│ 2.32  ┆ 2026-02-26   │
│ 1.7   ┆ 2026-01-30   │
│ 0.38  ┆ 2026-02-18   │
│ 3.1   ┆ 2026-01-26   │
└───────┴──────────────┘

Price Column Type : Float64


In [ ]:
from environment import DataCleaninEnv

# Initialize the gym with our master data

env = DataCleaninEnv("../data/raw/mind1_chaos_master.csv")

# See what the agent sees at the start(State)
initial_state = env.get_state()
print("---INITIAL STATE---(ROW [0])")
print(f"Price : {initial_state['price']}")
print(f"Date: {initial_state['date_unfixed']}")

# 3. Take an Action : Lets tell the agent to clean the price
# action_id 0 is the 'strip_currency'
reward, done = env.step(action_id=0, column_name="price")

# check  the result 
print("\n---AFTER ACTION---(step 1)")
print(f"Reward Received : {reward}")
print(f"\n is training finished? : {done}")

# 5 Check the next state (The gym should have moved to row 1)
next_state = env.get_state()
print(f"\nNext Row Price: {next_state['price']}")

---INITIAL STATE---(ROW [0])
